# SigLIP2 feature extraction on Kaggle

Bật **GPU** và **Internet**, sửa `DATASET_HANDLE` rồi Run all. Notebook tải đúng 44 version bằng KaggleHub và tự hợp nhất cấu trúc lồng `L21_V001/L21_V001/*.jpg` mà không sao chép ảnh. Dense features không được sinh trong batch toàn kho.

In [ ]:
from pathlib import Path

REPOSITORY_URL = "https://github.com/Maylyn-2day/AIC2026-Multimedia-Agent.git"
REPOSITORY_BRANCH = "Duc/DP_OI"
DATASET_HANDLE = "KAGGLE_USERNAME/DATASET_NAME"
DATASET_VERSIONS = range(1, 45)
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
MERGED_KEYFRAME_DIRECTORY = Path("/kaggle/working/keyframes")
OUTPUT_DIRECTORY = Path("/kaggle/working/siglip2")
EXPECTED_VIDEO_COUNT = 873
BATCH_SIZE = 4

In [ ]:
import subprocess
import sys

repository = Path("/kaggle/working/AIC2026-Multimedia-Agent")
if not repository.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(repository)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch==3.3.0", "huggingface_hub", "kagglehub", "safetensors"], check=True)
sys.path.insert(0, str(repository))

In [ ]:
import re
import shutil
import numpy as np
import torch
import kagglehub
from huggingface_hub import hf_hub_download
from backend.offline_indexing.feature_extractor import extract_siglip2_features

if not torch.cuda.is_available():
    raise RuntimeError("GPU chưa được bật: Kaggle Settings > Accelerator > GPU")
if DATASET_HANDLE == "KAGGLE_USERNAME/DATASET_NAME":
    raise ValueError("Sửa DATASET_HANDLE thành owner/dataset-name")
version_roots = [Path(kagglehub.dataset_download(f"{DATASET_HANDLE}/versions/{version}")) for version in DATASET_VERSIONS]

pattern = re.compile(r"L\d+_V\d+")
sources = {}
for root in version_roots:
    for directory in root.glob("**/L*_V*"):
        if not directory.is_dir() or not pattern.fullmatch(directory.name) or not any(directory.glob("*.jpg")):
            continue
        if directory.name in sources and sources[directory.name] != directory:
            raise ValueError(f"Trùng video_id {directory.name}: {sources[directory.name]} và {directory}")
        sources[directory.name] = directory
if len(sources) != EXPECTED_VIDEO_COUNT:
    raise ValueError(f"Cần {EXPECTED_VIDEO_COUNT} video nhưng tìm thấy {len(sources)}; kiểm tra đủ 44 Inputs")

MERGED_KEYFRAME_DIRECTORY.mkdir(parents=True, exist_ok=True)
for video_id, source in sources.items():
    link = MERGED_KEYFRAME_DIRECTORY / video_id
    if not link.exists():
        link.symlink_to(source, target_is_directory=True)

global_output = OUTPUT_DIRECTORY / "global"
global_output.mkdir(parents=True, exist_ok=True)
for previous in KAGGLE_INPUT_ROOT.glob("**/global/L*_V*.npy"):
    destination = global_output / previous.name
    if not destination.exists():
        shutil.copy2(previous, destination)

weights = Path(hf_hub_download("timm/ViT-gopt-16-SigLIP2-384", "open_clip_model.safetensors"))
video_ids = sorted(sources)
pending = []
for video_id in video_ids:
    image_count = sum(1 for _ in (MERGED_KEYFRAME_DIRECTORY / video_id).glob("*.jpg"))
    output = OUTPUT_DIRECTORY / "global" / f"{video_id}.npy"
    if output.exists() and np.load(output, mmap_mode="r").shape == (image_count, 1536):
        print(f"skip {video_id}: đã hoàn thành")
    else:
        pending.append(video_id)

print(f"GPU: {torch.cuda.get_device_name(0)} | videos: {len(video_ids)} | pending: {len(pending)}")
if pending:
    extract_siglip2_features(MERGED_KEYFRAME_DIRECTORY, OUTPUT_DIRECTORY, pending, BATCH_SIZE, "cuda", False, weights)

In [ ]:
import json
import shutil

reports = []
for video_id in video_ids:
    feature = np.load(OUTPUT_DIRECTORY / "global" / f"{video_id}.npy", mmap_mode="r")
    if feature.shape[1] != 1536 or not np.isfinite(feature).all():
        raise ValueError(f"Feature không hợp lệ: {video_id} {feature.shape}")
    reports.append({"video_id": video_id, "shape": list(feature.shape), "dtype": str(feature.dtype)})
manifest = {"model": "ViT-gopt-16-SigLIP2-384", "normalized": True, "videos": reports}
(OUTPUT_DIRECTORY / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(reports, indent=2))
archive = shutil.make_archive("/kaggle/working/siglip2-global", "zip", OUTPUT_DIRECTORY)
print(f"Download: {archive}")